<a href="https://colab.research.google.com/github/EshikaAbbaraju/Role_Specified_Collective_Foraging_Task_Model/blob/main/Role_Specified_Collective_Foraging_Task_Model_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!git clone https://github.com/JerryGuo2001/Role_Speficifed_Collective_Foraging_Task.git
#%cd Role_Speficifed_Collective_Foraging_Task

#tune the danger from aliens
#print out snapshot labels at the top

%cd /content/Role_Speficifed_Collective_Foraging_Task
CSV_DIR = "gridworld/"

Cloning into 'Role_Speficifed_Collective_Foraging_Task'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 305 (delta 21), reused 35 (delta 11), pack-reused 255 (from 1)
Receiving objects: 100% (305/305), 1.84 MiB | 13.47 MiB/s, done.
Resolving deltas: 100% (169/169), done.
/content/Role_Speficifed_Collective_Foraging_Task


In [18]:
# -*- coding: utf-8 -*-
"""
Collective foraging + security simulation.

Two agents alternate in discrete "rounds":
  - Round 1 → Forager moves `moves_per_round` steps (security frozen)
  - Round 2 → Security moves `moves_per_round` steps (forager frozen)
  - Repeat until total rounds are exhausted.

Only one agent acts per timestep. Mine depletion, visited cells, and
agent positions persist across all rounds (no world reset between rounds).
A GIF is produced showing both agents from an initial frame onward.
"""
from __future__ import annotations

from collections import deque
from dataclasses import dataclass, replace
from typing import Dict, List, Optional, Set, Tuple

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# Directory containing CSV environment files; falls back to current dir if absent
CSV_DIR = "gridworld/" if os.path.isdir("gridworld") else "."


def _normalize_identity(s: Optional[str]) -> str:
    """
    Canonicalise a user-supplied identity string to one of three internal tags:
      'leader'   – agent acts as a leader (positive λ, higher explore bias for forager;
                   fixes lead-move branch for security)
      'follower' – agent acts as a follower (negative λ, lower explore bias; follow branch)
      'auto'     – default: let the model formulas decide at runtime
    Raises ValueError for unrecognised inputs.
    """
    if s is None:
        return "auto"
    x = str(s).strip().lower()
    if x in ("lead", "leader", "l"):
        return "leader"
    if x in ("follow", "follower", "f"):
        return "follower"
    if x in ("auto", "a", "", "model", "formula"):
        return "auto"
    raise ValueError(f"identity must be 'leader', 'follower', or 'auto', got {s!r}")


# =============================================================================
# Environment
# =============================================================================
def load_env_from_csv(
    csv_path: str,
    prob_A: float = 0.8,   # reward probability for Gold Mine A (high richness)
    prob_B: float = 0.5,   # reward probability for Gold Mine B (medium richness)
    prob_C: float = 0.2,   # reward probability for Gold Mine C (low richness)
) -> pd.DataFrame:
    """
    Read a gridworld CSV and construct the environment DataFrame.

    Expected CSV columns: x (row), y (col), mine_type, alien_id.

    Returns a DataFrame indexed by (Row, Col) with columns:
      - Location   : "row:col" string key
      - mine_type  : raw mine label or "" for empty cells
      - richness   : "high" | "medium" | "low" | "none"
      - reward_prob: float probability of yielding a reward when dug
      - alien_level: integer threat level (0 = safe)
    """
    raw = pd.read_csv(csv_path)
    # Rename spatial columns so the rest of the code uses consistent names
    raw = raw.rename(columns={"x": "Row", "y": "Col", "mine_type": "mine_type", "alien_id": "alien_id"})

    rows = []
    for _, row in raw.iterrows():
        r, c = int(row["Row"]), int(row["Col"])
        mine_type = str(row["mine_type"]).strip() if not pd.isna(row["mine_type"]) else ""
        alien_val = row["alien_id"]

        # Map mine label to reward probability and richness tier
        if mine_type == "Gold Mine A":
            reward_prob, richness = prob_A, "high"
        elif mine_type == "Gold Mine B":
            reward_prob, richness = prob_B, "medium"
        elif mine_type == "Gold Mine C":
            reward_prob, richness = prob_C, "low"
        else:
            # Non-mine cell: no reward
            mine_type, reward_prob, richness = "", 0.0, "none"

        # alien_level 0 means no threat; NaN / empty string also maps to 0
        alien_level = 0 if (pd.isna(alien_val) or str(alien_val).strip() == "") else int(float(alien_val))

        rows.append({
            "Row": r, "Col": c, "Location": f"{r}:{c}",
            "mine_type": mine_type, "richness": richness,
            "reward_prob": float(reward_prob), "alien_level": alien_level,
        })

    env = pd.DataFrame(rows)
    # Multi-level index (Row, Col) allows fast O(1) lookups by grid position
    env.set_index(["Row", "Col"], inplace=True, drop=False)
    return env


# =============================================================================
# Forager
# =============================================================================
@dataclass
class ForagerConfig:
    """Hyper-parameters for the ForagerAgent."""
    moves_per_round: int = 5          # Steps per forager turn-block
    lambda_sec: float = 1.0           # Security-distance weight in move scoring (sign sets role)
    w_scale: float = 1.0              # Steepness of the sigmoid blending exploit vs explore
    move_cost: float = 0.0            # Penalty subtracted from move value (keeps agent moving)
    mine_capacity_high: int = 8       # Max digs from a Gold Mine A before depletion
    mine_capacity_medium: int = 5     # Max digs from a Gold Mine B
    mine_capacity_low: int = 2        # Max digs from a Gold Mine C
    security_pos: Optional[Tuple[int, int]] = None  # Override for starting security position
    seed: Optional[int] = None        # RNG seed for reproducibility
    # "auto" = use formulas; "leader"/"follower" adjusts λ sign and explore bias
    role_mode: str = "auto"


class ForagerAgent:
    """
    Adaptive forager that balances exploitation (digging known mines) and
    exploration (moving toward unvisited cells) using a sigmoid-weighted policy.

    Decision per step:
      1. If stunned (alien encounter), skip and count down stun.
      2. If current cell has reward probability > 0, dig (collect resource).
      3. Otherwise compute w (explore weight) and move to the highest-scoring
         neighbouring cell, accounting for security distance via λ.
    """

    def __init__(self, env: pd.DataFrame, cfg: ForagerConfig):
        self.env = env
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)

        # Place forager at the centre of the grid
        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self._start_pos = (max_r // 2, max_c // 2)
        self.pos = self._start_pos

        # Security position is shared so the forager can avoid / follow it
        self.security_pos = cfg.security_pos or (max_r // 2, max_c // 2)

        self.t = 0                        # Global step counter (forager steps only)
        self.total_reward = 0.0           # Cumulative reward across all rounds
        self.round_reward = 0.0           # Reward within the current segment (resets each forager turn)
        self.visited: Set[Tuple[int, int]] = {self.pos}  # Cells the forager has ever occupied
        self.stunned_steps = 0            # Remaining steps the forager is immobilised

        self.log: List[dict] = []         # Per-step structured log entries
        self.frames_action: List[str] = []  # Sequence of action strings for GIF annotation
        self.prev_pos: Optional[Tuple[int, int]] = None  # Position before last step (hints security lead)
        self.current_round: int = 0       # Which alternating turn-block is active

        # Initialise per-mine dig counters from config capacities
        self.mine_remaining: Dict[Tuple[int, int], int] = {}
        self.initial_mine_remaining: Dict[Tuple[int, int], int] = {}
        for (r, c), row in self.env.iterrows():
            mt = str(row["mine_type"]).strip()
            if mt == "Gold Mine A":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_high)
            elif mt == "Gold Mine B":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_medium)
            elif mt == "Gold Mine C":
                self.initial_mine_remaining[(r, c)] = int(cfg.mine_capacity_low)
        self.mine_remaining = dict(self.initial_mine_remaining)

    # ------------------------------------------------------------------
    # Core value functions
    # ------------------------------------------------------------------

    def current_reward_prob(self) -> float:
        """Return dig reward probability at current cell (0 if mine is depleted)."""
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] <= 0:
            return 0.0  # Mine exhausted
        return float(self.env.loc[self.pos, "reward_prob"])

    def neighbors(self) -> List[Tuple[int, int]]:
        """Return valid 4-connected neighbours (bounded by grid index)."""
        r, c = self.pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def Vdig(self) -> float:
        """Value of staying and digging = current cell's reward probability."""
        return self.current_reward_prob()

    def Vmove(self) -> float:
        """
        Value of moving = average reward rate so far this segment.
        Returns 0 on the very first step to avoid division by zero.
        """
        return self.round_reward / float(self.t) if self.t > 0 else 0.0

    def w_t(self, vd: float, vm: float) -> float:
        """
        Sigmoid blend weight that controls exploit vs explore:
          w → 0  : exploit (vd >> vm, staying is better)
          w → 1  : explore (vm >> vd, moving is better)
        w_scale sets the steepness; larger = sharper transition.
        """
        delta = vd - vm
        return float(1.0 / (1.0 + np.exp(-max(self.cfg.w_scale, 1e-6) * delta)))

    # ------------------------------------------------------------------
    # Goal attraction functions (used in move scoring)
    # ------------------------------------------------------------------

    def E_exploit(self, s_prime: Tuple[int, int]) -> float:
        """
        Exploitation attraction toward s_prime.
        Measured as best known mine reward_prob / (1 + Manhattan distance to nearest known mine).
        Returns 0 if no mines have been discovered yet.
        """
        known = [p for p in self.visited if p in self.env.index and self.env.loc[p, "mine_type"] != ""]
        if not known:
            return 0.0
        d_min = min(abs(s_prime[0] - p[0]) + abs(s_prime[1] - p[1]) for p in known)
        return max(float(self.env.loc[p, "reward_prob"]) for p in known) / (1.0 + d_min)

    def E_explore(self, s_prime: Tuple[int, int]) -> float:
        """
        Exploration attraction toward s_prime.
        Measured as 1 / (1 + Manhattan distance to nearest unvisited cell).
        Returns 0 when all cells have been visited.
        """
        unv = [p for p in self.env.index if p not in self.visited]
        if not unv:
            return 0.0
        d_min = min(abs(s_prime[0] - p[0]) + abs(s_prime[1] - p[1]) for p in unv)
        return 1.0 / (1.0 + d_min)

    def A_goal(self, s_prime: Tuple[int, int], w: float) -> float:
        """
        Combined goal attraction for candidate cell s_prime:
          A = (1 - w) * E_exploit + w * E_explore
        w comes from w_t (sigmoid) so the forager smoothly transitions
        between exploitation and exploration as conditions change.
        """
        return (1.0 - w) * self.E_exploit(s_prime) + w * self.E_explore(s_prime)

    # ------------------------------------------------------------------
    # Identity / role adjustments
    # ------------------------------------------------------------------

    def _lambda_for_moves(self) -> float:
        """
        λ modulates how security distance influences move scoring:
          leader  → positive λ: prefer cells *farther* from security (scout ahead)
          follower→ negative λ: prefer cells *closer* to security (stay near guard)
          auto    → use cfg.lambda_sec as-is (may be positive or negative)
        """
        lam = float(self.cfg.lambda_sec)
        rm = str(self.cfg.role_mode).lower()
        if rm in ("follow", "follower"):
            return -abs(lam) if lam != 0 else -1.0
        if rm in ("lead", "leader"):
            return abs(lam) if lam != 0 else 1.0
        return lam

    def _w_identity_adjust(self, w: float) -> float:
        """
        Shift w up (leader → more exploration) or down (follower → more exploitation)
        by a fixed ±0.2 offset, clamped to [0, 1].
        """
        rm = str(self.cfg.role_mode).lower()
        if rm in ("lead", "leader"):
            return float(min(1.0, w + 0.2))
        if rm in ("follow", "follower"):
            return float(max(0.0, w - 0.2))
        return w

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _log(self, **kw):
        """Append a structured log entry with sensible defaults for omitted fields."""
        kw.setdefault("step", self.t)
        kw.setdefault("row", self.pos[0])
        kw.setdefault("col", self.pos[1])
        kw.setdefault("total_reward", self.total_reward)
        kw.setdefault("round_reward", self.round_reward)
        kw.setdefault("round", self.current_round)
        kw.setdefault("forager_role_mode", self.cfg.role_mode)
        self.log.append(kw)

    def reset_segment_reward(self):
        """
        Reset the per-segment reward counter at the start of each forager turn-block.
        Does NOT move agents or reset mine capacities.
        """
        self.round_reward = 0.0

    def _dig(self) -> float:
        """
        Collect reward at the current cell:
          - Adds reward_prob to totals (stochastic reward simplified to expected value).
          - Decrements the mine's remaining capacity.
        Returns the reward gained.
        """
        r = self.current_reward_prob()
        self.total_reward += r
        self.round_reward += r
        # Deplete mine by one dig
        if self.pos in self.mine_remaining and self.mine_remaining[self.pos] > 0:
            self.mine_remaining[self.pos] -= 1
        return r

    def _move_to_best_A(self, w: float) -> Tuple[bool, int]:
        """
        Score all neighbours by:
          score(p) = λ * dist(p, security) * A_goal(p, w) − move_cost

        Move to the highest-scoring neighbour. If the chosen cell has a
        non-zero alien level AND security is not already there, the forager
        is stunned for 3 steps.

        Returns (moved: bool, alien_level_encountered: int).
        """
        nbrs = self.neighbors()
        if not nbrs:
            return False, 0

        best_val, best_p = -np.inf, None
        lam = self._lambda_for_moves()
        for p in nbrs:
            a = self.A_goal(p, w)
            d = abs(p[0] - self.security_pos[0]) + abs(p[1] - self.security_pos[1])
            v = lam * float(d) * a - self.cfg.move_cost
            if v > best_val:
                best_val, best_p = v, p

        if best_p is None:
            return False, 0

        self.pos = best_p
        self.visited.add(self.pos)

        # Check for alien encounter at new cell
        al = int(self.env.loc[self.pos, "alien_level"])
        if al > 0 and self.pos != self.security_pos:
            # Security presence neutralises the alien; otherwise forager is stunned
            self.stunned_steps = 3
        return True, al

    # ------------------------------------------------------------------
    # Public step interface
    # ------------------------------------------------------------------

    def step(self) -> str:
        """
        Execute one forager step and return an action label:
          'stunned'  – forager is immobilised (alien encounter recovery)
          'dig'      – forager collects reward at current cell
          'move'     – forager moves to a better cell
          'no_move'  – forager wanted to move but had nowhere to go

        Increments self.t after each call.
        """
        pos_before = tuple(self.pos)

        # --- Stun recovery: skip action, count down ---
        if self.stunned_steps > 0:
            self._log(action="stunned", decision="stunned", Vdig=np.nan, Vmove=np.nan, w=np.nan)
            self.frames_action.append("stunned")
            self.stunned_steps -= 1
            self.t += 1
            self.prev_pos = pos_before
            return "stunned"

        # --- Dig branch: current cell has reward available ---
        if self.current_reward_prob() > 0.0:
            vd, vm = self.Vdig(), self.Vmove()
            w = self._w_identity_adjust(self.w_t(vd, vm))
            r_gained = self._dig()
            self._log(action="dig", decision="dig", Vdig=vd, Vmove=vm, w=w, r_gained=r_gained)
            self.frames_action.append("dig")
            self.t += 1
            self.prev_pos = pos_before
            return "dig"

        # --- Move branch: no reward here, consider moving ---
        vd, vm = self.Vdig(), self.Vmove()
        w = self._w_identity_adjust(self.w_t(vd, vm))

        # Force a move if any neighbour has reward OR unexplored cells remain,
        # to avoid the forager being stuck on an empty cell by a marginal Vdig > Vmove
        force_move = (
            self.current_reward_prob() == 0.0
            and (
                any(self.env.loc[p, "reward_prob"] > 0 for p in self.neighbors())
                or any(p not in self.visited for p in self.env.index)
            )
        )

        if (vd > vm) and not force_move:
            # Dig even though reward_prob == 0 is theoretically possible via this branch;
            # in practice force_move above prevents this for sensible environments.
            self._dig()
            self._log(action="dig", decision="dig", Vdig=vd, Vmove=vm, w=w)
            self.frames_action.append("dig")
        else:
            moved, alien_lvl = self._move_to_best_A(w)
            self._log(
                action="move" if moved else "no_move",
                decision="move",
                Vdig=vd, Vmove=vm, w=w,
                alien_level=alien_lvl if alien_lvl else None,
            )
            self.frames_action.append("move" if moved else "no_move")

        self.t += 1
        self.prev_pos = pos_before
        return self.frames_action[-1]


# =============================================================================
# Security
# =============================================================================
@dataclass
class SecurityConfig:
    """Hyper-parameters for the SecurityAgent."""
    vtype_threshold: float = 0.5      # Vtype above this → prefer lead role
    gamma: float = 0.15               # Learning rate for updating lead/follow weights
    lead_weight_init: float = 0.5     # Initial weight favouring lead moves
    follow_weight_init: float = 0.5   # Initial weight favouring follow moves
    window_size: int = 10             # Rolling window length for Vmove averaging
    e_scan: float = 1.0               # Scan efficiency multiplier for Vscan
    follow_radius: int = 2            # Max Manhattan distance when orbiting forager (follow mode)
    strategy_if_tie: str = "follow"   # Tiebreaker when Vmove_lead == Vmove_follow
    seed: Optional[int] = None
    # "auto" = formula-driven; "leader"/"follower" hard-wires the move branch
    role_mode: str = "auto"


class SecurityAgent:
    """
    Security agent that patrols the gridworld to neutralise aliens and protect the forager.

    Each security step:
      1. MOVE: choose between chasing the nearest alien (if within 3 cells),
               leading the forager (step toward predicted next forager cell), or
               orbiting the forager within follow_radius (follow mode).
      2. SCAN: compute Vscan at the new position (logged, does not override movement).

    Role selection (lead vs follow) is driven by Vtype – a composite score
    measuring distance, area revealed, and stun rate – updated online each step.
    """

    def __init__(self, env, cfg: SecurityConfig, start_pos: Tuple[int, int]):
        self.env = env
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)
        self._start_pos = start_pos
        self.pos = start_pos

        # Dmax normalises Manhattan distances to [0, 1] across the grid
        max_r = int(self.env.index.get_level_values(0).max())
        max_c = int(self.env.index.get_level_values(1).max())
        self.Dmax = max(1, max_r + max_c)

        # Adaptive weights for lead vs follow (updated via online learning)
        self.lead_weight = float(cfg.lead_weight_init)
        self.follow_weight = float(cfg.follow_weight_init)

        # Rolling history of Vmove estimates under each role
        self.vmove_lead_hist: deque = deque(maxlen=cfg.window_size)
        self.vmove_follow_hist: deque = deque(maxlen=cfg.window_size)

        self.log: List[dict] = []
        self._prev_security_pos: Optional[Tuple[int, int]] = None  # Used to prevent U-turns

    # ------------------------------------------------------------------
    # Navigation helpers
    # ------------------------------------------------------------------

    def _neighbors(self, pos: Tuple[int, int]) -> List[Tuple[int, int]]:
        """Return valid 4-connected neighbours of pos."""
        r, c = pos
        return [p for p in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)] if p in self.env.index]

    def _manhattan(self, a: Tuple[int, int], b: Tuple[int, int]) -> int:
        """Manhattan distance between two grid cells."""
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def _step_toward(self, target: Tuple[int, int], exclude: Optional[Tuple[int, int]] = None) -> bool:
        """
        Move one step toward target along the Manhattan-optimal path.
        Optionally exclude a cell (e.g., previous position) to avoid U-turns.
        Returns True if a move was made.
        """
        if self.pos == target:
            return False
        nbrs = self._neighbors(self.pos)
        if exclude is not None:
            filt = [p for p in nbrs if p != exclude]
            if filt:
                nbrs = filt  # Only avoid previous cell if alternatives exist
        if not nbrs:
            return False
        self.pos = min(nbrs, key=lambda p: self._manhattan(p, target))
        return True

    def _orbit_near(
        self,
        target: Tuple[int, int],
        radius: int,
        exclude: Optional[Tuple[int, int]] = None,
    ) -> bool:
        """
        Follower patrol: maintain proximity to target within `radius` cells.

        If already within radius: prefer moving to cells at distance 1 from target
        (ring-orbit rather than standing still). Stable sort on coordinates prevents
        oscillation between symmetric options.

        If outside radius: step toward target to re-enter orbit.

        Returns True if a move was made.
        """
        raw = self._neighbors(self.pos)
        if not raw:
            return False

        # Outside orbit radius → re-approach
        if self._manhattan(self.pos, target) > radius:
            return self._step_toward(target, exclude=exclude)

        # Inside radius → circulate along the ring
        cands = [p for p in raw if exclude is None or p != exclude]
        if not cands:
            cands = raw

        in_ring = [p for p in cands if self._manhattan(p, target) <= radius]
        pool = in_ring if in_ring else cands

        # Primary key: distance from ring edge (prefer distance-1 from target);
        # secondary keys: row, col for a deterministic, anti-oscillation tiebreak
        self.pos = min(
            pool,
            key=lambda p: (abs(self._manhattan(p, target) - 1), p[0], p[1]),
        )
        return True

    # ------------------------------------------------------------------
    # Alien / mine scanning helpers
    # ------------------------------------------------------------------

    def _alien_prob_at(self, pos: Tuple[int, int]) -> float:
        """
        Composite alien encounter probability at pos:
          p = 0.7 * reward_prob + 0.3 * min(1, alien_level / 3)
        Higher-value mine cells and high alien-level cells both increase risk.
        Clamped to [0, 1].
        """
        base_reward = float(self.env.loc[pos, "reward_prob"])
        alien_level = float(self.env.loc[pos, "alien_level"]) if "alien_level" in self.env.columns else 0.0
        p = 0.7 * base_reward + 0.3 * min(1.0, alien_level / 3.0)
        return float(np.clip(p, 0.0, 1.0))

    def _gold_mines_around(self, center: Tuple[int, int], radius: int = 2) -> int:
        """Count all mines (any type) within `radius` Manhattan steps of center."""
        cr, cc = center
        n = 0
        for (r, c), row in self.env.iterrows():
            if abs(r - cr) + abs(c - cc) <= radius and str(row["mine_type"]).strip() != "":
                n += 1
        return n

    def _nearest_alien_cell(self, from_pos: Tuple[int, int]) -> Optional[Tuple[int, int]]:
        """Return the grid cell with alien_level > 0 closest to from_pos, or None if none exist."""
        cells = [(r, c) for (r, c), row in self.env.iterrows() if int(row["alien_level"]) > 0]
        if not cells:
            return None
        return min(cells, key=lambda p: self._manhattan(from_pos, p))

    # ------------------------------------------------------------------
    # Vtype: composite role-selection score
    # ------------------------------------------------------------------

    def compute_vtype(
        self,
        forager_pos: Tuple[int, int],
        revealed_count: int,
        total_cells: int,
        stunned_count_total: int,
        global_step: int,
    ) -> Dict[str, float]:
        """
        Compute Vtype_t ∈ [0, 1], a composite signal used to decide lead vs follow:

          distance_norm_t  = Manhattan(forager, security) / Dmax
          area_revealed_t  = |visited cells| / |all cells|
          stun_rate_t      = cumulative stuns / (global_step + 1)
          Vtype_t          = distance_norm * area_revealed * (1 − stun_rate)

        Interpretation:
          High Vtype → forager is far away and has revealed a lot but is not often stunned
                       → security should lead (stay ahead of the forager).
          Low  Vtype → forager is nearby / few cells explored / frequently stunned
                       → security should follow (guard closely).

        Returns a dict with all component values for logging.
        """
        distance_t = self._manhattan(forager_pos, self.pos)
        distance_norm_t = float(np.clip(distance_t / self.Dmax, 0.0, 1.0))
        area_revealed_t = float(np.clip(revealed_count / max(1, total_cells), 0.0, 1.0))
        stun_rate_t = float(np.clip(stunned_count_total / max(1, global_step + 1), 0.0, 1.0))
        vtype_t = distance_norm_t * area_revealed_t * (1.0 - stun_rate_t)
        vtype_t = float(np.clip(vtype_t, 0.0, 1.0))
        return {
            "distance_t": distance_t,
            "distance_norm_t": distance_norm_t,
            "area_revealed_t": area_revealed_t,
            "stun_rate_t": stun_rate_t,
            "vtype_t": vtype_t,
        }

    # ------------------------------------------------------------------
    # Online weight update
    # ------------------------------------------------------------------

    def _update_weights(self):
        """
        Online learning rule: compare rolling-average Vmove under each role.
        The weight for the better-performing role increases by γ × performance gap;
        the other weight decreases symmetrically (they always sum to ≤ 1).
        Skips update if either history is empty (early steps).
        """
        if not self.vmove_lead_hist or not self.vmove_follow_hist:
            return
        avg_lead = float(np.mean(self.vmove_lead_hist))
        avg_follow = float(np.mean(self.vmove_follow_hist))
        g = self.cfg.gamma
        if avg_lead > avg_follow:
            self.lead_weight = min(1.0, self.lead_weight + g * (avg_lead - avg_follow))
            self.follow_weight = max(0.0, 1.0 - self.lead_weight)
        elif avg_follow > avg_lead:
            self.follow_weight = min(1.0, self.follow_weight + g * (avg_follow - avg_lead))
            self.lead_weight = max(0.0, 1.0 - self.follow_weight)

    # ------------------------------------------------------------------
    # Public step interface
    # ------------------------------------------------------------------

    def step(
        self,
        t_in_round: int,
        forager_agent: ForagerAgent,
        round_idx: int,
        global_step: int,
        forager_pos_start: Tuple[int, int],
        forager_pos_before: Optional[Tuple[int, int]],
    ) -> Tuple[str, float]:
        """
        Execute one security step (forager is frozen during this turn-block).

        Args:
            t_in_round        : Step index within the current security turn-block (0-based).
            forager_agent     : Reference to the forager (for visited cells, stun count, etc.).
            round_idx         : Global alternating-round index.
            global_step       : Absolute step counter across all agents.
            forager_pos_start : Forager's current (frozen) position.
            forager_pos_before: Forager's position before its last move (velocity hint for lead).

        Returns:
            (move_action: str, vscan_t: float)
            move_action describes what the security agent did; vscan_t is the scan value.
        """
        prev_before_move = tuple(self.pos)   # Remember position for U-turn avoidance
        exclude = self._prev_security_pos    # Cell to avoid stepping back into immediately

        forager_pos = forager_pos_start
        revealed_count = len(forager_agent.visited)
        total_cells = len(self.env.index)
        stunned_total = sum(1 for x in forager_agent.frames_action if x == "stunned")

        # --- Compute Vtype and role-selection signals ---
        m = self.compute_vtype(forager_pos, revealed_count, total_cells, stunned_total, global_step)
        vtype_t = m["vtype_t"]
        role_by_vtype = "lead" if vtype_t >= self.cfg.vtype_threshold else "follow"

        # Estimate forager mobility near its current position
        p_alien_here = self._alien_prob_at(self.pos)
        gold_mines_around = self._gold_mines_around(forager_pos, radius=2)
        inferred_forager_movement = gold_mines_around * (1.0 - p_alien_here)
        vmove_base = inferred_forager_movement * p_alien_here * vtype_t

        # Compute per-role Vmove estimates using adaptive weights + distance bias
        lead_bonus = self.lead_weight * (1.0 + m["distance_norm_t"])
        follow_bonus = self.follow_weight * (1.0 + (1.0 - m["distance_norm_t"]))
        vmove_lead_t = vmove_base * lead_bonus
        vmove_follow_t = vmove_base * follow_bonus

        # Update rolling history and re-learn weights
        self.vmove_lead_hist.append(vmove_lead_t)
        self.vmove_follow_hist.append(vmove_follow_t)
        self._update_weights()

        # Select role by comparing estimated Vmove values
        if vmove_lead_t > vmove_follow_t:
            role_by_choice = "lead"
        elif vmove_follow_t > vmove_lead_t:
            role_by_choice = "follow"
        else:
            role_by_choice = self.cfg.strategy_if_tie  # Tiebreaker from config

        # Fall back to Vtype-based role when Vmove estimates are indistinguishable
        final_role = role_by_choice if abs(vmove_lead_t - vmove_follow_t) > 1e-9 else role_by_vtype

        # --- Override with hard-wired role if role_mode is set ---
        role_forced = False
        rm = str(self.cfg.role_mode).lower()
        if rm in ("lead", "leader"):
            final_role, role_forced = "lead", True
        elif rm in ("follow", "follower"):
            final_role, role_forced = "follow", True

        # --- Determine movement priority ---
        nearest_alien = self._nearest_alien_cell(self.pos)
        # Chase aliens that are within 3 cells (immediate threat)
        chase = nearest_alien is not None and self._manhattan(self.pos, nearest_alien) <= 3

        moved = False
        move_action = "idle"

        if chase and nearest_alien is not None:
            # Priority 1: Chase the nearest alien
            moved = self._step_toward(nearest_alien, exclude=exclude)
            move_action = "move_chase_alien" if moved else "chase_no_step"

        elif final_role == "lead":
            # Priority 2 (lead mode): Extrapolate forager's next position from velocity
            cur = forager_pos
            if forager_pos_before is not None:
                dr = cur[0] - forager_pos_before[0]
                dc = cur[1] - forager_pos_before[1]
                predicted = (cur[0] + dr, cur[1] + dc)
                if predicted not in self.env.index:
                    predicted = cur  # Predicted cell outside grid → stay on current
            else:
                predicted = cur  # No velocity hint; head to forager's current cell

            moved = self._step_toward(predicted, exclude=exclude)
            move_action = "move_lead" if moved else "lead_no_step"

        else:
            # Priority 2 (follow mode): Orbit within follow_radius of forager
            moved = self._orbit_near(forager_pos, self.cfg.follow_radius, exclude=exclude)
            move_action = "move_follow" if moved else "follow_no_step"

        # --- Fallback: if still not moved and not at forager, step toward forager ---
        if not moved and self.pos != forager_pos:
            if self._step_toward(forager_pos, exclude=exclude):
                move_action = move_action + "+fallback_to_forager"
                moved = True

        # --- Scan current (post-move) cell ---
        # Vscan = scan efficiency * alien encounter probability at new position
        p_alien_after = self._alien_prob_at(self.pos)
        vscan_t = self.cfg.e_scan * p_alien_after

        # delta_t: how much lead dominates over follow at this step (for logging)
        delta_t = float(np.clip(self.lead_weight - self.follow_weight, 0.0, 1.0))

        # Record previous position for next step's U-turn avoidance
        self._prev_security_pos = prev_before_move

        self.log.append({
            "global_step": global_step,
            "round": round_idx,
            "step_in_round": t_in_round,
            "security_row": self.pos[0],
            "security_col": self.pos[1],
            "security_move": move_action,
            "security_role": final_role,
            "role_forced": role_forced,
            "Vscan_t": vscan_t,
            "distance_t": m["distance_t"],
            "distance_norm_t": m["distance_norm_t"],
            "area_revealed_t": m["area_revealed_t"],
            "stun_rate_t": m["stun_rate_t"],
            "Vtype_t": vtype_t,
            "Vmove_lead_t": vmove_lead_t,
            "Vmove_follow_t": vmove_follow_t,
            "delta_t": delta_t,
            "lead_weight": self.lead_weight,
            "follow_weight": self.follow_weight,
        })

        return move_action, vscan_t


# =============================================================================
# Frame buffer + run
# =============================================================================
@dataclass
class AnimFrame:
    """Snapshot of world state captured after each step, used to build the GIF."""
    turn_idx: int        # Alternating round index (0-based); -1 for the initial frame
    phase: str           # "forager" | "security" | "—" (start frame)
    step_in_phase: int   # Step within the active agent's block; -1 on the start frame
    global_step: int     # Absolute step counter across all agents
    forager_pos: Tuple[int, int]
    security_pos: Tuple[int, int]
    forager_action: str
    security_move: str
    security_role: str
    vscan: float
    depleted: Set[Tuple[int, int]]   # Mine cells with zero remaining capacity
    forager_total_reward: float
    forager_round_reward: float
    is_start: bool = False           # True only for the pre-simulation frame


def run_foraging_simulation(
    csv_path: str,
    rounds: int = 3,
    forager_cfg: Optional[ForagerConfig] = None,
    security_cfg: Optional[SecurityConfig] = None,
    seed: int = 0,
    out_prefix: str = "forager",
    save_csv: bool = True,
    save_gif: bool = True,
    forager_identity: str = "auto",
    security_identity: str = "auto",
) -> Dict[str, object]:
    """
    Run the full alternating-round simulation.

    Turn structure:
      Even turn_idx (0, 2, 4, …) → forager moves `moves_per_round` steps
      Odd  turn_idx (1, 3, 5, …) → security moves `moves_per_round` steps
    Total primitive steps = rounds × moves_per_round (plus one start GIF frame).

    Args:
        csv_path          : Path to the gridworld CSV.
        rounds            : Number of alternating turn-blocks to simulate.
        forager_cfg       : ForagerConfig (defaults used if None).
        security_cfg      : SecurityConfig (defaults used if None).
        seed              : RNG seed applied to both agents when their configs lack one.
        out_prefix        : Filename prefix for CSV and GIF outputs.
        save_csv          : Whether to write forager/security log CSVs.
        save_gif          : Whether to write the animated GIF.
        forager_identity  : "auto" | "leader" | "follower" – sets forager role_mode.
        security_identity : "auto" | "leader" | "follower" – sets security role_mode.

    Returns:
        Dict containing env, agent objects, logs, frames, and summary statistics.
    """
    env = load_env_from_csv(csv_path)
    forager_cfg = forager_cfg or ForagerConfig(seed=seed)
    security_cfg = security_cfg or SecurityConfig(seed=seed)

    # Normalise identity strings and inject them into configs (immutable dataclass → replace)
    fi = _normalize_identity(forager_identity)
    si = _normalize_identity(security_identity)
    forager_cfg = replace(forager_cfg, role_mode=fi)
    security_cfg = replace(security_cfg, role_mode=si)

    # Both agents start at the grid centre
    max_r = int(env.index.get_level_values(0).max())
    max_c = int(env.index.get_level_values(1).max())
    start = (max_r // 2, max_c // 2)

    forager = ForagerAgent(env, forager_cfg)
    security = SecurityAgent(env, security_cfg, start_pos=start)
    # Share security position so the forager can factor it into move scoring
    forager.security_pos = security.pos

    frames: List[AnimFrame] = []
    moves = forager_cfg.moves_per_round
    num_turn_rounds = rounds
    global_step = 0

    # --- Initial GIF frame: show both agents before any action ---
    frames.append(
        AnimFrame(
            turn_idx=-1,
            phase="—",
            step_in_phase=-1,
            global_step=-1,
            forager_pos=tuple(forager.pos),
            security_pos=tuple(security.pos),
            forager_action="(start)",
            security_move="(start)",
            security_role="—",
            vscan=0.0,
            depleted=set(),
            forager_total_reward=forager.total_reward,
            forager_round_reward=forager.round_reward,
            is_start=True,
        )
    )

    # --- Main simulation loop: alternate between forager and security turn-blocks ---
    for turn_idx in range(num_turn_rounds):
        forager_turn = (turn_idx % 2 == 0)   # Even indices → forager
        forager.current_round = turn_idx

        if forager_turn:
            # Forager's turn: security is frozen (position doesn't change)
            for k in range(moves):
                # Reset segment reward counter at the start of each new forager block
                if k == 0 and turn_idx > 0:
                    forager.reset_segment_reward()

                # Update forager's view of security position before each step
                forager.security_pos = tuple(security.pos)
                fa = forager.step()

                # Record which mines are now depleted for rendering
                depleted = {p for p, rem in forager.mine_remaining.items() if rem <= 0}
                frames.append(
                    AnimFrame(
                        turn_idx=turn_idx,
                        phase="forager",
                        step_in_phase=k,
                        global_step=global_step,
                        forager_pos=tuple(forager.pos),
                        security_pos=tuple(security.pos),  # Frozen
                        forager_action=fa,
                        security_move="(frozen)",
                        security_role="—",
                        vscan=0.0,
                        depleted=set(depleted),
                        forager_total_reward=forager.total_reward,
                        forager_round_reward=forager.round_reward,
                        is_start=False,
                    )
                )
                global_step += 1

        else:
            # Security's turn: forager is frozen (position stays at end of its last block)
            for k in range(moves):
                forager_pos_start = tuple(forager.pos)  # Forager frozen here
                forager_prev = forager.prev_pos          # Velocity hint for lead extrapolation

                security.step(
                    t_in_round=k,
                    forager_agent=forager,
                    round_idx=turn_idx,
                    global_step=global_step,
                    forager_pos_start=forager_pos_start,
                    forager_pos_before=forager_prev,
                )

                sec_row = security.log[-1]   # Latest log entry from security step
                depleted = {p for p, rem in forager.mine_remaining.items() if rem <= 0}
                frames.append(
                    AnimFrame(
                        turn_idx=turn_idx,
                        phase="security",
                        step_in_phase=k,
                        global_step=global_step,
                        forager_pos=tuple(forager.pos),    # Frozen
                        security_pos=tuple(security.pos),
                        forager_action="(frozen)",
                        security_move=str(sec_row["security_move"]),
                        security_role=str(sec_row["security_role"]),
                        vscan=float(sec_row["Vscan_t"]),
                        depleted=set(depleted),
                        forager_total_reward=forager.total_reward,
                        forager_round_reward=forager.round_reward,
                        is_start=False,
                    )
                )
                global_step += 1

    forager_log = pd.DataFrame(forager.log)
    security_log = pd.DataFrame(security.log)

    if save_csv:
        forager_log.to_csv(f"{out_prefix}_forager.csv", index=False)
        security_log.to_csv(f"{out_prefix}_security.csv", index=False)

    if save_gif:
        animate_simulation(
            env,
            frames,
            num_turn_rounds=rounds,
            moves_per_round=forager_cfg.moves_per_round,
            outpath=f"{out_prefix}.gif",
        )

    return {
        "env": env,
        "forager": forager,
        "security": security,
        "forager_log": forager_log,
        "security_log": security_log,
        "frames": frames,
        "total_reward": forager.total_reward,
        "rounds": rounds,
        "moves_per_round": forager_cfg.moves_per_round,
        "total_agent_steps": global_step,
        "forager_identity": fi,
        "security_identity": si,
    }


def animate_simulation(
    env: pd.DataFrame,
    frames: List[AnimFrame],
    num_turn_rounds: int,
    moves_per_round: int,
    outpath: str = "forager.gif",
):
    """
    Render the simulation as an animated GIF.

    Each AnimFrame → one GIF frame. Depleted mines are shown as NaN (gray).
    Forager and security markers are offset slightly when co-located so both are visible.
    """
    env_idx = env.set_index(["Row", "Col"], drop=False)
    max_r, max_c = int(env_idx["Row"].max()), int(env_idx["Col"].max())

    # Base reward-probability grid used as the background heatmap
    base = np.zeros((max_r + 1, max_c + 1))
    for (r, c), row in env_idx.iterrows():
        base[r, c] = row["reward_prob"]

    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = plt.cm.YlOrRd.copy()
    cmap.set_bad(color="lightgray")   # NaN (depleted) cells shown as gray
    im = ax.imshow(base, cmap=cmap, origin="upper", vmin=0, vmax=1)

    # Forager marker: blue circle
    fd, = ax.plot(
        [], [],
        "o",
        color="tab:blue",
        markersize=11,
        markeredgecolor="white",
        markeredgewidth=1.5,
        label="Forager",
        zorder=5,
    )
    # Security marker: green square
    sd, = ax.plot(
        [], [],
        "s",
        color="limegreen",
        markersize=10,
        markeredgecolor="black",
        markeredgewidth=1.2,
        label="Security",
        zorder=6,
    )

    ax.set_xticks(range(max_c + 1))
    ax.set_yticks(range(max_r + 1))
    ax.grid(True, linestyle=":", alpha=0.3)
    ax.legend(loc="upper right")

    def init():
        """Initialise artists before animation starts; shows both agents from frame 0."""
        im.set_data(base)
        if frames:
            f0 = frames[0]
            r0, c0 = f0.forager_pos
            rs, cs = f0.security_pos
            # If agents share a cell, offset markers vertically so both are visible
            if (r0, c0) == (rs, cs):
                r0p, rsp = r0 - 0.22, rs + 0.22
            else:
                r0p, rsp = r0, rs
            fd.set_data([c0], [r0p])
            sd.set_data([cs], [rsp])
        else:
            fd.set_data([], [])
            sd.set_data([], [])
        ax.set_title("")
        return [im, fd, sd]

    def update(i):
        """Update all artists for frame i."""
        fr = frames[i]

        # Overlay depleted mines as NaN so they render as gray
        g = base.copy()
        for dr, dc in fr.depleted:
            g[dr, dc] = np.nan
        im.set_data(g)

        # Place agent markers, offsetting if co-located
        frg, fcg = fr.forager_pos
        sr, sc = fr.security_pos
        if (frg, fcg) == (sr, sc):
            frg_plot, sr_plot = frg - 0.22, sr + 0.22
        else:
            frg_plot, sr_plot = frg, sr
        fd.set_data([fcg], [frg_plot])
        sd.set_data([sc], [sr_plot])

        # Build title strings with context-dependent formatting
        gstep = "—" if fr.global_step < 0 else str(fr.global_step + 1)
        stext = "—" if fr.step_in_phase < 0 else f"{fr.step_in_phase + 1}/{moves_per_round}"

        if fr.is_start:
            title = (
                f"Alternating rounds: each round = {moves_per_round} steps by ONE agent (no world reset)\n"
                f"INITIAL  |  global step {gstep}  |  next: ROUND 1 = FORAGER ×{moves_per_round}\n"
                f"Forager: {fr.forager_action}  |  Security: {fr.security_move}\n"
                f"Reward total={fr.forager_total_reward:.3f}  segment_reward={fr.forager_round_reward:.3f}"
            )
        else:
            if fr.phase == "forager":
                active = "FORAGER moving"
                sec_extra = ""
            elif fr.phase == "security":
                active = "SECURITY moving"
                # Show security role and scan value when it is the active agent
                sec_extra = f" ({fr.security_role})  |  Vscan={fr.vscan:.3f}"
            else:
                active = str(fr.phase)
                sec_extra = ""

            frozen_note = (
                "other agent frozen"
                if fr.phase in ("forager", "security")
                else ""
            )
            title = (
                f"ROUND {fr.turn_idx + 1}/{num_turn_rounds}  |  {active}  |  step {stext}  |  global {gstep}  |  {frozen_note}\n"
                f"Forager: {fr.forager_action}  |  Security: {fr.security_move}{sec_extra}\n"
                f"Reward total={fr.forager_total_reward:.3f}  segment_reward={fr.forager_round_reward:.3f}"
            )
        ax.set_title(title, fontsize=9)
        return [im, fd, sd]

    # blit=False required because both the title text and the image data change each frame
    anim = FuncAnimation(
        fig,
        update,
        init_func=init,
        frames=len(frames),
        interval=450,    # ms between frames (≈ 3 fps when saved with PillowWriter)
        blit=False,
        repeat=True,
    )
    anim.save(outpath, writer=PillowWriter(fps=3))
    plt.close(fig)


if __name__ == "__main__":
    # Quick smoke-test: run a 20-round simulation on a single environment CSV
    single_csv = os.path.join(CSV_DIR, "middle_reward_middle_risk_01.csv")
    if not os.path.isfile(single_csv):
        print(f"Skip run: CSV not found: {single_csv}")
    else:
        out = run_foraging_simulation(
            single_csv,
            rounds=20,
            forager_cfg=ForagerConfig(moves_per_round=5, seed=0),
            security_cfg=SecurityConfig(seed=0),
            out_prefix="forager_sim",
        )
        print("Total reward:", out["total_reward"])
        print("Frames:", len(out["frames"]))

Total reward: 14.000000000000002
Frames: 101
